<a href="https://colab.research.google.com/github/ProgrammerNesi/llm-engineering/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install the libraries required to load and run an open-source LLM with 4-bit quantization.

In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.1 MB/s eta 0:00:00


Import the tokenizer, model loader, quantization configuration, and JSON library for saving the generated dataset.

In [ ]:
import json
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

Choose the open-source LLM you want to use. Since we use AutoTokenizer and AutoModel, changing only this string lets us switch to another compatible model.

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"

Load the model in 4-bit precision to reduce memory usage while maintaining almost the same performance.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

Download the tokenizer and pretrained model from Hugging Face. The model is automatically loaded onto the available GPU.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=quant_config
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Instead of asking random questions, we create a reusable prompt template. Changing only the topic allows us to generate datasets for different domains.

In [ ]:
topic = "Transformers"

prompt = f"""
Generate 10 diverse question-answer pairs about {topic}.

Return ONLY valid JSON.

Format:

[
  {{
      "question":"",
      "answer":""
  }}
]
"""

Convert the prompt into the format expected by the chat model, tokenize it into token IDs, convert them into a PyTorch tensor, and move the tensor to the GPU.

In [ ]:
messages = [
    {
        "role":"user",
        "content":prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

Run inference. The Transformer repeatedly predicts one token at a time until the response is complete.

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=1000
)
print(outputs)

KeyboardInterrupt: 

The model returns token IDs. The tokenizer converts those IDs back into readable text.

In [ ]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)
print(outputs)
print(response)

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    271,  31115,    220,     16,
             15,  16807,   3405,  95777,  13530,    911,  80532,    382,   5598,
          26687,   2697,   4718,    382,   4061,   1447,   9640,    220,    341,
            414,    330,   7841,  42398,    414,    330,   9217,   3252,    698,
            220,    456,    921, 151645,    198, 151644,  77091,    198,   9640,
            220,    341,    414,    330,   7841,   3252,   3838,    374,    279,
           6028,    729,    315,    264,  62379,    304,   5662,   6832,  35718,
            414,    330,   9217,   3252,   8963,    388,    525,  15503,   1483,
            369,   8500,    311,   8500,   9079,    323,   5810,   4128,   8692,
            320,     45,  12567,      8,   8357,   4152,    311,    862,   5726,
            311,  12322,   1